# S15 · Forecast a series with ARIMA

An air-conditioner brand wants to know how many units it will sell next month, so
it can stock its warehouses. All it has is the past monthly sales. Today we take a
series like that, find the pattern in it, and continue the story forward a couple
of years, with an honest band showing how sure we are.

Along the way we meet **ARIMA**, the classic forecasting model, and the plain idea
underneath it: today usually leans on the recent past.

**New here? Read this once.**

- New to Python? You can still do this whole notebook. Press the play button on
  each cell, top to bottom, and read the plain-English note above each one.
- New to time series? Open the primer
  `primers/time_series_and_forecasting.md` for a ten-minute, picture-first version.
- Already confident with code or with forecasting? Skip ahead to the cells marked
  **Stretch (optional)**.
- Stuck on a word? It is in `primers/glossary.md`.

## Setup

On **Google Colab**, run the next cell once. On your **own machine** you already
installed everything with `uv`, so it does nothing there.

In [ ]:
# This notebook uses statsmodels for the forecasting parts.
# Colab usually has it, but we install it to be safe. Nothing runs off Colab.
import sys
if "google.colab" in sys.modules:
    !pip install -q statsmodels
else:
    print("Not on Colab - assuming the libraries are already installed.")

In [ ]:
import numpy as np                     # fast maths on lists of numbers
import pandas as pd                      # tables and dated series
import matplotlib.pyplot as plt          # drawing charts

## A time series, in one sentence

A **time series** is just a column of numbers recorded at regular times: monthly
sales, daily prices, hourly load. **Forecasting** means guessing the next few
numbers using only the past ones.

Most real series are the sum of three simple things:

- a **trend**, a slow drift up or down (the brand is growing),
- a **season**, a pattern that repeats every fixed number of steps (AC sales spike
  every summer, so a 12-month cycle),
- **noise**, the random wobble we cannot explain.

We will *build* such a series ourselves, so we know the true answer, then see if
ARIMA can recover the pattern.

## Step 1 — build a monthly demand series (trend + season + noise)

We make 120 monthly sales figures for a made-up product. The trend rises slowly
(the brand grows), the season repeats every 12 months (a yearly summer peak), and
we add a little random noise so it is not too neat. We set a seed so everyone gets
the same numbers.

In [ ]:
# Set a seed so we all get the same random numbers every time.
np.random.seed(0)

# 120 time steps, thought of as 120 months.
number_of_months = 120
time = np.arange(number_of_months)

# The three ingredients, added together.
trend = 20 + 0.18 * time                            # slow upward drift
season = 5 * np.sin(2 * np.pi * time / 12)          # repeats every 12 months
noise = np.random.normal(0, 1.5, number_of_months)  # random wobble

monthly_sales = trend + season + noise

# Give the series a real monthly date index so it looks like data from a system.
dates = pd.date_range("2010-01-01", periods=number_of_months, freq="MS")
series = pd.Series(monthly_sales, index=dates)

# Look at the first few months.
print(series.head())

## Step 2 — always plot the series first

Before any modelling, look at the data. You should be able to *see* the slow rise
(the trend) and the repeating up-and-down wiggle (the season).

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(series.index, series.values, color="#2E75B6")
plt.xlabel("month")
plt.ylabel("units sold")
plt.title("Made-up monthly demand (trend + season + noise)")
plt.show()

## Step 3 — split the parts apart

`statsmodels` can split a series back into trend, season and noise in one call.
This confirms what our eyes already saw, and builds the habit: look at the parts
before choosing a method. If there is no season, you will not add a seasonal
term later.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

# period=12 tells it the season repeats every 12 months.
parts = seasonal_decompose(series, model="additive", period=12)

# Draw the four panels: the series, its trend, its season, and the leftover noise.
figure = parts.plot()
figure.set_size_inches(9, 7)
plt.show()

## Step 4 — why we must 'difference' the series

ARIMA wants a **stationary** series: one whose average and spread stay roughly the
same over time. Ours is not stationary, because the trend keeps pushing the
average up.

The fix is **differencing**: instead of the value itself, look at the *change* from
one month to the next. A rising trend turns into a roughly flat series of changes.
We confirm it with the **ADF test**, a quick check that returns a small p-value
(below 0.05) when a series looks stationary.

In [ ]:
from statsmodels.tsa.stattools import adfuller

# The change from each month to the next. .diff() subtracts the previous value;
# the first entry has no "previous", so we drop it with .dropna().
differenced = series.diff().dropna()

# The ADF test returns several numbers; the second one is the p-value.
adf_on_original = adfuller(series)
adf_on_differenced = adfuller(differenced)

print("ADF p-value, original series   :", round(adf_on_original[1], 3))
print("ADF p-value, differenced series:", round(adf_on_differenced[1], 3))
print()
print("A p-value below 0.05 means the series looks stationary.")

## Step 5 — see the difference

The original series drifts upward. Its differenced version hovers around a flat
level. That flatness is exactly what ARIMA wants.

In [ ]:
# Two plots stacked, sharing the same time axis.
figure, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)

axes[0].plot(series.index, series.values, color="#C0392B")
axes[0].set_title("Original series (trending - not stationary)")
axes[0].set_ylabel("units sold")

axes[1].plot(differenced.index, differenced.values, color="#27AE60")
axes[1].axhline(differenced.mean(), color="grey", linestyle="--")
axes[1].set_title("After one difference (flat-ish - stationary)")
axes[1].set_ylabel("month-to-month change")
axes[1].set_xlabel("month")

plt.tight_layout()
plt.show()

## What AR and MA mean, in plain words

ARIMA is built from two ideas, and both are just a **weighted average over time**:

- **AR (autoregressive)** says *today leans on recent days*. Today is a weighted
  sum of the last few values. Think momentum: yesterday nudges today.
- **MA (moving average)** says *today leans on recent surprises*. Today is a
  weighted sum of the last few random shocks. Think of a fading echo of the last
  surprise.

That is the whole heart of it. If you want the exact formulas, they are in the
optional box near the end. To use ARIMA you only pick three small whole numbers,
which we do next.

### Optional — what (p, d, q) mean

ARIMA(`p`, `d`, `q`) is three small whole numbers:

- `p` — how many recent **values** today leans on (the AR part),
- `d` — how many times we differenced to reach stationarity (we differenced once,
  so `d = 1`),
- `q` — how many recent **shocks** today leans on (the MA part).

So ARIMA(1, 1, 1) means: difference once, then use one past value and one past
shock. Small numbers are usually best. Skip this box if you like; the next cell
just uses (1, 1, 1).

## Step 6 — fit an ARIMA model

We pick a small, fast order, ARIMA(1, 1, 1), and let `statsmodels` find the best
weights for us. Fitting is a one-liner. The skill is in choosing the order and
then checking the result, which we do right after.

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

# order = (p, d, q) = (1 past value, difference once, 1 past shock).
arima_model = ARIMA(series, order=(1, 1, 1))
arima_fit = arima_model.fit()

# The summary prints the fitted weights and some quality scores.
print(arima_fit.summary())

## Step 7 — check the residuals (the most-skipped step)

**Residuals** are what the model could not explain: actual minus predicted. If the
model is good, the residuals should look like pure noise, with no trend and no
leftover pattern, centred on zero. If they still have shape, the model missed
something. This check is not optional in real work, even though it is the step
everyone is tempted to skip.

In [ ]:
# The leftovers the model could not explain.
residuals = arima_fit.resid

plt.figure(figsize=(9, 4))
plt.plot(residuals.index, residuals.values, color="#7D3C98")
plt.axhline(0, color="grey", linestyle="--")
plt.xlabel("month")
plt.ylabel("residual (actual - predicted)")
plt.title("Residuals should look like random noise around zero")
plt.show()

## Step 8 — a formal residual test (Ljung-Box)

The eyeball check has a formal version, the **Ljung-Box test**. Here a *large*
p-value (above 0.05) is the good outcome: it means "no leftover pattern detected",
so the residuals look like noise.

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox

# Test whether the residuals still have a pattern up to lag 10.
ljung_box = acorr_ljungbox(residuals, lags=[10], return_df=True)

print(ljung_box)
print()
print("A p-value (lb_pvalue) above 0.05 means the residuals look like noise - good.")

## Step 9 — forecast ahead, with a prediction interval

Now the payoff. We ask the model for the next 24 months. We also ask for a
**prediction interval**, a band showing how uncertain each forecast is. A forecast
is a *fan*, not a line: the band widens the further ahead we look, because the
future gets harder to know. For the warehouse team, that band is the useful part.

In [ ]:
# Forecast the next 24 months.
number_of_steps_ahead = 24
forecast = arima_fit.get_forecast(steps=number_of_steps_ahead)

# The single best-guess line...
forecast_mean = forecast.predicted_mean

# ...and the 95% band around it (a lower and an upper column).
confidence_band = forecast.conf_int(alpha=0.05)

print("First few forecast values:")
print(forecast_mean.head())

## Step 10 — plot the history, the forecast, and the band

The blue line is the past we have seen, the red line is the forecast, and the
shaded area is the 95% prediction interval. Notice how it fans out.

In [ ]:
plt.figure(figsize=(10, 5))

# The history we already have.
plt.plot(series.index, series.values, color="#2E75B6", label="history")

# The forecast line.
plt.plot(forecast_mean.index, forecast_mean.values,
         color="#C0392B", label="forecast")

# The shaded prediction interval. The two columns are the lower and upper edges.
lower_edge = confidence_band.iloc[:, 0]
upper_edge = confidence_band.iloc[:, 1]
plt.fill_between(forecast_mean.index, lower_edge, upper_edge,
                 color="#C0392B", alpha=0.2, label="95% interval")

plt.xlabel("month")
plt.ylabel("units sold")
plt.title("ARIMA forecast with a widening prediction interval")
plt.legend()
plt.show()

### Stretch (optional) — does a bigger model forecast better?

Skip this if you are new to code. If you are comfortable, here is a habit worth
having: bigger is not automatically better. We fit a larger ARIMA(2, 1, 2) and
compare it to our ARIMA(1, 1, 1) using **AIC**, a score that rewards fit but
punishes extra complexity. Lower AIC is better. Often the small model wins, which
is why we start small.

In [ ]:
# Fit a larger model on the same series.
bigger_model = ARIMA(series, order=(2, 1, 2))
bigger_fit = bigger_model.fit()

print("AIC of ARIMA(1,1,1):", round(arima_fit.aic, 1))
print("AIC of ARIMA(2,1,2):", round(bigger_fit.aic, 1))
print()
print("Lower AIC is better. The extra parameters have to earn their place.")

## What you just did

You ran the full **Box-Jenkins** recipe: make the series stationary by
differencing, fit an ARIMA model, **check the residuals**, and forecast ahead with
an honest band. The two engines inside ARIMA, AR and MA, are both just weighted
averages over time: one over past values, one over past surprises.

Next notebook: `02_exponential_smoothing.ipynb`, a gentler second way to forecast
that often keeps right up with ARIMA.